In [ ]:
#| default_exp j

# iversonnb j
> Running J in-process through libj, and `j` magics for Jupyter and IPython

In [ ]:
import ctypes
from ctypes import c_void_p,c_char_p,c_int,c_longlong,c_double,c_byte,POINTER,byref,cast,create_string_buffer
from shutil import which
from IPython.display import display
from IPython.utils.capture import capture_output
from fastcore.utils import *
from fastcore.test import *

J's console, `jconsole`, is a thin wrapper around **libj**, the J engine shared library. Rather than scraping the console over a pipe, iversonnb loads libj directly: `JDo` runs one line of J and returns only when it's done, output arrives through a callback we register, and `JInterrupt` stops a long computation from another thread. The engine, its C API (`jsource/jsrc/jlib.h`), and `profile.ijs` (the standard library bootstrap) all ship with every J install, including `pip install jlanguage`. This notebook builds the binding up one step at a time.

## Finding J

In [ ]:
_LIBJ = 'j.dll' if sys.platform=='win32' else 'libj.dylib' if sys.platform=='darwin' else 'libj.so'

def find_j():
    "Locate the J binary directory: the one containing libj and profile.ijs"
    if (p:=which('jconsole')) and (d:=Path(p).resolve().parent/_LIBJ).exists(): return d.parent
    try:
        import jlang
        return Path(jlang.path())/'bin'
    except ImportError: pass
    roots = [Path.home(), Path('/Applications'), Path('/opt'), Path('/usr/share')]
    cands = sorted(c for r in roots if r.exists() for c in r.glob('j9*') if (c/'bin'/_LIBJ).exists())
    if cands: return cands[-1]/'bin'
    raise FileNotFoundError('J not found: pip install jlanguage, or install it from jsoftware.com')

A J install is a directory with `bin/` holding the engine library and `profile.ijs` side by side. `which` finds a real `jconsole` and resolves symlinks to its directory (the `pip install jlanguage` console script is a Python shim with no libj next to it, so the existence check skips it and the `jlang` import supplies the path instead). The fallbacks scan the standard per-OS install locations.

In [ ]:
find_j()

Path('/Users/jhoward/aai-ws/.venv/lib/python3.13/site-packages/jlang/bin')

## The engine

In [ ]:
_OUTCB = ctypes.CFUNCTYPE(None, c_void_p, c_int, c_void_p)   # Joutput(jt, type, text)
_INCB  = ctypes.CFUNCTYPE(c_void_p, c_void_p, c_char_p)      # Jinput(jt, prompt) -> line
_SMCON = 3   # smoptions: identify as a console front end

def load_j(jbin=None):
    "Load libj from `jbin` (or `find_j()`), declare its C signatures, and return `(jbin,lib)`"
    jbin = Path(jbin or find_j())
    lib = ctypes.CDLL(str(jbin/_LIBJ))
    lib.JInit2.restype = c_void_p
    lib.JInit2.argtypes = [c_char_p]
    lib.JDo.argtypes = [c_void_p,c_char_p]
    lib.JSM.argtypes = [c_void_p,c_void_p]
    lib.JInterrupt.argtypes = lib.JFree.argtypes = [c_void_p]
    lib.JGetM.argtypes = lib.JSetM.argtypes = [c_void_p,c_char_p]+[POINTER(c_longlong)]*4
    return jbin,lib

The whole API is in `jsource/jsrc/jlib.h`: `JInit2` returns an engine instance told where its own directory is (bare `JInit` exists too, but then the engine can't find `libgmp` next to itself, so extended precision breaks -- and mixing the two in one process can crash), `JDo` runs one line of J against it, and `JSM` registers a 5-slot callback array `{output, wd, input, unused, options}` (`jconsole` passes exactly `{Joutput, 0, Jinput, 0, SMCON}`). The output callback receives everything J writes, tagged with a type: 1 is a formatted result, 2 an error display, 5 an exit request from `2!:55` (the text pointer carries the exit code, so it's declared `c_void_p` and decoded per type -- it can be NULL). The input callback supplies continuation lines when a multi-line definition is open. Here's the smallest working setup, callbacks that just collect into a list:

In [ ]:
jbin,lib = load_j()
jt = lib.JInit2(str(jbin).encode())
out = []
@_OUTCB
def outcb(j,typ,p): out.append((typ, ctypes.string_at(p).decode('utf-8','replace') if p else p))
cbs = (c_void_p*5)(cast(outcb,c_void_p), None, None, None, _SMCON)
lib.JSM(jt, cbs)
lib.JDo(jt, b'+/ % # 1 2 3 4'), out

(0, [(1, '0.25\n')])

`JDo` returns 0 for success (the J error number otherwise), and by the time it returns, the result has already arrived through the callback: completion detection is free, no prompt-scraping or sentinel needed. But this is a bare engine: no standard library. `jconsole` boots one by running a first sentence that sets `BINPATH` and `ARGV` and then runs `profile.ijs` (see `jefirst` in `jsource/jsrc/jeload.c`); we do the same. J executes right to left, so `BINPATH` is set first:

In [ ]:
out.clear()
boot = f"(3 : '0!:0 y')<BINPATH,'/profile.ijs'[ARGV_z_=:<'jconsole'[BINPATH_z_=:'{jbin}'"
lib.JDo(jt, boot.encode()), out

(0, [])

The stdlib is now loaded:

In [ ]:
out.clear()
lib.JDo(jt, b"toupper 'j is here'"), out

(0, [(1, 'J IS HERE\n')])

## Errors

Errors need no scraping either: `JDo` returns the J error number, and the error display arrives as type-2 output. `JError` carries the session's display as its message, like `AplError` does:

In [ ]:
out.clear()
lib.JDo(jt, b"1 + 'a'"), out

(3, [(2, "|domain error, executing dyad +\n|y is character\n|   1    +'a'\n")])

In [ ]:
class JError(Exception):
    "A J error, carrying the session's output (including the error display) as its message"

## A session object

`J` packages all of the above: load, init, register callbacks, boot the profile. The output callback collects `(type,text)` pairs, records exit requests, and guards against NULL. The input callback pops lines from a queue that `run` fills: J calls it whenever a multi-line definition is open, so a whole cell can be queued and each line lands where it belongs -- top-level lines via `JDo`, body lines via the callback. The returned buffer is kept alive on `self` because the engine reads it after the callback returns (returning a Python `bytes` directly would leak: ctypes can't manage the lifetime, and warns so).

In [ ]:
class J:
    "A J session: the libj engine loaded in-process, with the stdlib profile booted"
    def __init__(self, jbin=None):
        self.jbin,self._lib = load_j(jbin)
        self.jt = self._lib.JInit2(str(self.jbin).encode())
        self._out,self._inq,self.exited = [],[],None
        @_OUTCB
        def _o(j,typ,p):
            if typ==5: self.exited = p or 0
            else: self._out.append((typ, ctypes.string_at(p).decode('utf-8','replace') if p else ''))
        @_INCB
        def _i(j,prompt):
            self._inbuf = create_string_buffer((self._inq.pop(0) if self._inq else ')').encode())
            return ctypes.addressof(self._inbuf)
        self._cbs = (c_void_p*5)(cast(_o,c_void_p), None, cast(_i,c_void_p), None, _SMCON)
        self._lib.JSM(self.jt, self._cbs)
        boot = f"(3 : '0!:0 y')<BINPATH,'/profile.ijs'[ARGV_z_=:<'jconsole'[BINPATH_z_=:'{self.jbin}'"
        if self._lib.JDo(self.jt, boot.encode()): raise JError(''.join(t for _,t in self._out))

`run` queues the cell's lines and feeds each remaining top-level line to `JDo` (a definition's body lines are consumed from the queue by the input callback in between). Any nonzero return raises `JError` with everything the session printed, error display included; otherwise that output is returned:

In [ ]:
@patch
def run(self:J, code):
    "Run `code` (one or more lines of J) in the session, returning its output; raises `JError` on J errors"
    self._out.clear()
    self._inq[:] = code.strip().splitlines()
    while self._inq and self.exited is None:
        rc = self._lib.JDo(self.jt, self._inq.pop(0).encode())
        if rc and self.exited is None: raise JError(''.join(t for _,t in self._out))
    if self.exited is not None: return ''.join(t for ty,t in self._out if ty!=2)   # drop the exit unwind noise
    return ''.join(t for _,t in self._out)

In [ ]:
j = J()
print(j.run('m =: 2 3 $ 10 * 1 + i. 6\nm'))

10 20 30
40 50 60



State persists across calls, multi-line definitions work (the body lines travel through the input callback), and errors raise `JError` with the session's error display:

In [ ]:
test_eq(j.run('+/ , m'), '210\n')
test_eq(j.run('mean =: 3 : 0\n(+/ y) % # y\n)\nmean 1 2 3 4'), '2.5\n')
test_fail(lambda: j.run("m + 'x'"), contains='domain error')

Wrapping every use in `print(j.run(...))` is clunky. `__call__` makes the session itself a function, and `JOut`'s verbatim `__repr__` means bare expressions display exactly as they do in a J session, while still comparing and slicing as ordinary strings. A call with no output returns None, so nothing displays:

In [ ]:
class JOut(str):
    "Output text from a `J` call; displays verbatim"
    def __repr__(self): return str(self)

@patch
def __call__(self:J, code):
    "Run `code`, returning session output (or None if there is none)"
    return JOut(self.run(code)) or None

In [ ]:
j('m ,. |. m')

10 20 30 40 50 60
40 50 60 10 20 30

In [ ]:
test_eq(j('+/ , m'), '210\n')
test_is(j('m2 =: 10 * m'), None)

Extended precision works because `JInit2` lets the engine find its bundled `libgmp`:

In [ ]:
j('*/ 1 + i. 25x')

15511210043330985984000000

## Getting values into Python

`run` returns display text: what you want to look at, not what you want to compute with. `JGetM` hands over a noun's actual data: type, rank, and pointers to shape and ravel, straight out of engine memory. The common types map directly (booleans and integers, floats, characters); anything else (boxed, extended, rational) raises. `_nest` folds the flat ravel back into the array's shape:

In [ ]:
def _nest(x, shape):
    "Nest flat sequence `x` (row-major ravel) into `shape`"
    if len(shape)<2: return x
    n = len(x)//shape[0]
    return [_nest(x[i*n:(i+1)*n], shape[1:]) for i in range(shape[0])]

@patch
def getm(self:J, name):
    "Read noun `name` into Python: str for characters; int/float scalars and (nested) lists otherwise"
    t,r,s,d = c_longlong(),c_longlong(),c_longlong(),c_longlong()
    if self._lib.JGetM(self.jt, name.encode(), byref(t),byref(r),byref(s),byref(d)): raise JError(f'no noun: {name}')
    shape = cast(s.value, POINTER(c_longlong))[:r.value]
    n = math.prod(shape)
    if t.value==2: return _nest(ctypes.string_at(d.value, n).decode('utf-8','replace'), shape)
    if t.value not in (1,4,8): raise JError(f'unsupported J type {t.value} for: {name}')
    x = cast(d.value, POINTER({1:c_byte,4:c_longlong,8:c_double}[t.value]))[:n]
    return _nest(x, shape) if r.value else x[0]

`pyval` evaluates an arbitrary expression by assigning it to a scratch noun, reading that, and erasing it. Square brackets read better still:

In [ ]:
@patch
def pyval(self:J, expr):
    "Evaluate `expr` and return the result as a Python value"
    self.run(f'jnbtmp =: {expr}')
    try: return self.getm('jnbtmp')
    finally: self.run("4!:55 <'jnbtmp'")

@patch
def __getitem__(self:J, expr): return self.pyval(expr)

In [ ]:
test_eq(j['m'], [[10,20,30],[40,50,60]])
test_eq(j['m > 25'], [[0,0,1],[1,1,1]])
test_eq(j['+/ % # 1 2 3 4'], 0.25)
test_eq(j["'py' , 'val'"], 'pyval')

`JSetM` is the same interface in reverse, so Python values can go into the workspace too: `_jdat` builds the typed buffers (strings as characters; numbers as integers, or floats if any element is one), and `__setitem__` ships them. `fn` rounds the ergonomics out, turning any J verb into a Python callable -- one argument applies it monadically, two dyadically, in J's left-argument-first order:

In [ ]:
def _flat(o): return [a for x in o for a in _flat(x)] if isinstance(o,(list,tuple)) else [o]

def _jdat(v):
    "ctypes `(type,shape,buf)` for a Python scalar, string, or uniformly nested list"
    if isinstance(v,str):
        b = v.encode()
        return 2, [len(b)], create_string_buffer(b, len(b))
    shape,x = [],v
    while isinstance(x,(list,tuple)): shape,x = shape+[len(x)],x[0]
    flat = _flat(v)
    if any(isinstance(a,float) for a in flat): return 8, shape, (c_double*len(flat))(*flat)
    return 4, shape, (c_longlong*len(flat))(*flat)

@patch
def __setitem__(self:J, nm, v):
    "Assign Python value `v` (scalar, string, or nested list) to noun `nm`"
    t,shape,buf = _jdat(v)
    sh = (c_longlong*len(shape))(*shape)
    a = [c_longlong(t), c_longlong(len(shape)), c_longlong(ctypes.addressof(sh)), c_longlong(ctypes.addressof(buf))]
    if self._lib.JSetM(self.jt, nm.encode(), *map(byref,a)): raise JError(f'JSetM failed: {nm}')

In [ ]:
j['q'] = [[1,2],[3,4.5]]
test_eq(j['q'], [[1,2],[3,4.5]])
j['s'] = "it's"
test_eq(j['s'], "it's")
test_eq(j['+/ , q'], 10.5)

In [ ]:
@patch
def fn(self:J, code):
    "A Python callable applying J verb `code` monadically or dyadically (left argument first)"
    def f(*args):
        if len(args)==1:
            self['jnby'] = args[0]
            return self.pyval(f'({code}) jnby')
        self['jnbx'],self['jnby'] = args
        return self.pyval(f'jnbx ({code}) jnby')
    return f

In [ ]:
sq = j.fn('*:')
test_eq(sq([1,2,3]), [1,4,9])
test_eq(j.fn('+/')([1,2,3]), 6)
test_eq(j.fn('{.')(2, [5,6,7]), [5,6])

## Interrupting

`JDo` blocks its calling thread at the C level, so a Python signal handler can't run while J computes -- whoever hosts a `J` session must call `interrupt` from *another* thread (that's exactly what the jkernel worker does with SIGINT). It's safe cross-thread: it just sets the engine's break flag, which J polls between sentences, so the running line stops with an attention interrupt error and the session survives:

In [ ]:
@patch
def interrupt(self:J):
    "Stop the currently running sentence with an attention interrupt; safe to call from another thread"
    self._lib.JInterrupt(self.jt)

In [ ]:
import threading,time

In [ ]:
j('spin =: 3 : 0\nn =. 0\nwhile. n < 1e8 do. n =. n + 1 end.\n)')
t0 = time.time()
threading.Timer(0.5, j.interrupt).start()
test_fail(lambda: j('spin 0'), contains='attention interrupt')
assert time.time()-t0 < 5
test_eq(j('2+2'), '4\n')

## Exit requests and shutdown

When J code asks to leave (`exit 0`, or the `2!:55` foreign it wraps), the request arrives as a type-5 output whose text pointer carries the exit code. The engine itself keeps going -- it's the front end's decision -- so `J` records the code in `.exited`, `run` returns quietly instead of raising, and a host like the jkernel can honor it. Unlike a Dyalog session there's no child process to orphan: the engine lives and dies with the host process, so no `atexit` handling is needed and `close` (or the context manager) exists just to free the instance's memory explicitly. A closed session is unusable.

In [ ]:
@patch
def close(self:J):
    "Free the engine instance; the session is unusable afterwards"
    self._lib.JFree(self.jt)
    self.jt = None

@patch
def __enter__(self:J): return self

@patch
def __exit__(self:J, *args): self.close()

In [ ]:
with J() as j2:
    test_is(j2.exited, None)
    j2('exit 7')
    test_eq(j2.exited, 7)

## The `j` magics

`%%j` runs a cell and displays the session output verbatim; `%j expr` returns the expression's value as a Python object (it's just `j[expr]`), so it works on the right of an assignment: `z = %j z`. A trailing `;` on a cell suppresses its output, and the engine only starts on first use, not when the magic is registered. Loading the `iversonnb.j` extension registers it.

In [ ]:
class JMagic:
    "IPython `%j`/`%%j` magics, driving a lazily-started `J` session"
    def __init__(self, jbin=None): self.jbin,self.o = jbin,None

    def j(self, line, cell=None):
        "Run J: a cell magic displays the session output; a line magic returns the expression's Python value"
        if not self.o: self.o = J(self.jbin)
        if cell is None: return self.o[line.strip()]
        disp,cell = True,cell.rstrip()
        if cell.endswith(';'): disp,cell = False,cell[:-1]
        out = self.o(cell)
        if disp and out: display(out)

In [ ]:
def create_j_magic(shell=None):
    "Create a `JMagic` and register its `j` line/cell magic with `shell`, returning it"
    if not shell: shell = get_ipython()
    jm = JMagic()
    shell.register_magic_function(jm.j, 'line_cell', 'j')
    return jm

def load_ipython_extension(ipython):
    "Required function for creating magic"
    create_j_magic(shell=ipython)

In [ ]:
# Only required if you don't load the extension
magic = create_j_magic()

In [ ]:
%%j
m3 =: 3 3 $ i. 9
m3 +/ . * m3

15 18  21
42 54  66
69 90 111

The line magic brings values back into Python, and a trailing `;` suppresses cell output entirely:

In [ ]:
z = %j m3
test_eq(z, [[0,1,2],[3,4,5],[6,7,8]])

In [ ]:
%%j
big =: 1000 1000 $ i. 5
big + big;

In [ ]:
#| hide
with capture_output() as cap: magic.j('', '2+2;')
test_eq(len(cap.outputs), 0)
with capture_output() as cap: magic.j('', '2+2')
test_eq(len(cap.outputs), 1)

## Cleanup

Free the engines this notebook started: the magic's session, the `J` object's, and the raw walkthrough one.

In [ ]:
if magic.o: magic.o.close()
j.close()
lib.JFree(jt)

0

## Export -

In [ ]:
#|hide
#|eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()